# FEM Plate-with-Hole GNN Surrogate — Colab GPU setup

Sibling notebook to the AirfRANS project's `colab_setup.ipynb`, simplified: this
dataset (200 Abaqus cases, ~5-6k nodes each, ~207MB raw + ~78MB cached) is small
enough to just live in the git repo itself -- unlike AirfRANS's ~15GB dataset,
there's no external download step, no Drive mount needed for the data. All real
logic still lives in `src/`; this notebook is infrastructure only.

Runtime > Change runtime type > GPU, before running anything below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo

Colab VMs reset every session, so the repo needs to come from somewhere durable.
Pushed to `https://github.com/Revanthkr1/fem-plate-gnn` (**public**) --
`data/raw/*.json` and `data/norm_stats.npz` are committed, so they arrive with
the clone; only `data/cache/` (gitignored, rebuilt in section 3) is missing
after this. No token needed -- plain `git clone` works since the repo is public.

In [ ]:
REPO_URL = "https://github.com/Revanthkr1/fem-plate-gnn.git"

!git clone $REPO_URL repo
%cd repo

## 2. Install dependencies

Colab ships torch with CUDA already installed -- don't reinstall it.
`torch_geometric` installs as pure Python here too (same `torch_geometric.utils.scatter`-only
usage as AirfRANS's model.py, ported unchanged). No `airfrans` package needed --
this project doesn't depend on it.

In [ ]:
!pip install -q torch_geometric lightning pyvista pyyaml

## 3. Rebuild the cache

`data/raw/*.json` (400 cases as of phase 11: 200 single-hole + 200 with
variable hole count 0-3) and `data/norm_stats.npz` came with the clone.
`data/cache/` (preprocessed graph tensors) is gitignored and cheap to rebuild --
these are small meshes (~5-9k nodes depending on hole count, not AirfRANS's
~180k), so this is seconds, not the dominant cost VTU parsing was there.
Safe to re-run: `preprocess_case` skips any case already cached.

In [ ]:
import glob
import os

from src.preprocess import preprocess_split

RAW_DIR = "data/raw"
CACHE_DIR = "data/cache"

n_cases = len(glob.glob(os.path.join(RAW_DIR, "case_*.json")))
case_ids = list(range(n_cases))
preprocess_split(RAW_DIR, case_ids, CACHE_DIR)
print(f"cached {len(glob.glob(os.path.join(CACHE_DIR, 'case_*.pt')))}/{n_cases} cases")

## 4. Smoke test: one case on the real GPU (optional)

Same forward/backward timing check as the AirfRANS notebook's section 4 --
confirms the model + a real cached graph actually move to the GPU and run
before committing to a full training loop. Skip if you're just resuming a
training run.

In [ ]:
import time
import numpy as np
import torch

from src.dataset import CachedPyGPlateHoleDataset
from src.model import MeshGraphNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

stats = dict(np.load("data/norm_stats.npz"))
ds = CachedPyGPlateHoleDataset(CACHE_DIR, [0], stats=stats)
data = ds[0].to(device)

model = MeshGraphNet(node_in_dim=3, edge_in_dim=2, out_dim=3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
pred = model(data.x, data.edge_index, data.edge_attr)
loss = torch.nn.functional.mse_loss(pred, data.y)
loss.backward()
opt.step()
print(f"nodes={data.x.shape[0]}, edges={data.edge_index.shape[1]}, "
      f"forward+backward+step={time.time()-t0:.2f}s, loss={loss.item():.4f}")

## 5. Train

**Phase 11: the actual geometric-generalization test.** Cases 200-399 vary
hole *count* (0-3), not just hole radius/position within one template. Every
case with exactly 3 holes (51 cases) is held out of training *entirely* --
`src/splits.py::hole_count_splits()` -- so evaluating on them afterward tests
whether the model generalizes to a hole count it has genuinely never seen,
not just a new radius/position combination. A normal random 20-case
in-distribution holdout is drawn from the remaining (0/1/2-hole) cases, same
role as the phase 8-10b validation split. `data/norm_stats.npz` was
recomputed over the training-distribution cases only (excludes the held-out
count=3 bucket, so normalization can't leak any information about it).

Architecture is the phase-10b-validated one (`node_in_dim=3`, `[x, y, load]`
only -- no hand-fed hole geometry; `n_message_passing=8`) which beat every
previous result including the hand-engineered-feature version (3.9%
peak-stress error, 4.9mm location error on its own held-out split -- see
`PROJECT_FLOW.md` phase 10b). This run doesn't change the architecture,
just the training data and split.

**Own checkpoint subdirectory** (`DRIVE_ROOT/phase11/`), same reasoning as
every run since phase 9's collision bug: never reuse a directory another
experiment's checkpoints live in.

Checkpoints go to Drive so they survive a session reset. Once this finishes,
bring the checkpoint back for evaluation on *both* splits -- the normal
held-out set and, more importantly, the held-out count=3 cases -- since
that comparison is the actual point of this phase.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/fem-plate-gnn-data"
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
import yaml

from src.splits import hole_count_splits
from src.train import main as train_main

config = yaml.safe_load(open("configs/base.yaml"))

# Held-out count=3 cases never enter training at all -- that's the actual
# generalization test. n_val below matches len(splits["val"]) exactly, so
# train.py's existing case_ids[-n_val:] slicing lands on the right cases.
splits = hole_count_splits("data/raw", held_out_count=3, n_val=20, seed=0)
case_ids = splits["train"] + splits["val"]
print(f"train={len(splits['train'])}, val={len(splits['val'])}, "
      f"held-out count=3 (not used here)={len(splits['test_ood'])}")

# Own subdirectory, not just a different filename -- see section 5 markdown.
RUN_DIR = os.path.join(DRIVE_ROOT, "phase11")
os.makedirs(RUN_DIR, exist_ok=True)

train_main(
    cache_dir=CACHE_DIR,
    stats_path="data/norm_stats.npz",
    checkpoint_path=os.path.join(RUN_DIR, "meshgraphnet_phase11.ckpt"),
    case_ids=case_ids,
    model_kwargs=config["model"],
    max_epochs=config["training"]["max_epochs"],
    batch_size=config["training"]["batch_size"],
    accumulate_grad_batches=config["training"]["accumulate_grad_batches"],
    n_val=len(splits["val"]),
    lr=config["training"]["lr"],
    checkpoint_every_n_epochs=config["training"]["checkpoint_every_n_epochs"],
    num_workers=config["training"]["num_workers"],
    precision="16-mixed",  # GPU-specific override -- base.yaml's 32-true is for local CPU runs
)